# Kalshi 15-minute strategy experiment

Executed companion analysis for the isolated paper ledgers and strategy-lab wallets. Results are descriptive, fee-aware, and do not treat hindsight quotes as achievable fills.

## TL;DR

- The final main paper wallet closed 7 trades at 57.1% win rate and $-0.36 net P&L.
- Across 18 matched lab markets, adaptive exit captured 87.0% of winning-market hold upside; fixed exit captured 39.6%.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'research' / 'final_experiment_analysis.py').exists():
    raise RuntimeError('Execute this notebook from the repository root')
sys.path.insert(0, str(ROOT))
from research.final_experiment_analysis import (
    ExperimentRun, analyze_experiment
)
RUN_SPECS = [('baseline', 'data/simulations/20260824_final_2h_new_models_adaptive_exit'), ('consensus_reversal', 'data/simulations/20260824_2h_consensus_reversal_counterfactual')]
runs = [ExperimentRun(label, ROOT / directory) for label, directory in RUN_SPECS]
analysis = analyze_experiment(runs, ROOT / 'models/commodity_directional.json')
saved = json.loads((ROOT / 'reports/20260824_feature_confirmation/analysis_summary.json').read_text())
assert analysis['aggregate'] == saved['aggregate']
print('Independent recomputation matches saved aggregate metrics.')

Independent recomputation matches saved aggregate metrics.


## Context and methods

Main-paper trades are assigned to a run by its recorded UTC start/end bounds. Strategy-lab wallets are isolated SQLite files. Fixed and adaptive exits are compared only when a hold-to-settlement wallet entered the same ticker and side in the same session. Winning-market upside capture and losing-market loss avoidance are reported separately.

In [2]:
run_rows = []
for run in analysis['runs']:
    run_rows.append({
        'run': run['label'], 'status': run['run']['status'],
        'snapshots': run['market']['market_snapshots'],
        'depth_rows': run['market']['orderbook_levels'],
        'main_trades': run['main']['closed'],
        'main_win_rate': run['main']['win_rate'],
        'main_pnl': run['main']['net_pnl'],
        'quality_passed': run['quality']['passed'],
        'model_hash': run['run']['model_hash'],
        'config_hash': run['run']['config_hash'],
    })
pd.DataFrame(run_rows)

,run,status,snapshots,depth_rows,main_trades,main_win_rate,main_pnl,quality_passed,model_hash,config_hash
0,baseline,COMPLETED,8365,310118,6,1.000000,14.7162,True,bundle:5036382a936772fc01e50e9a36a2fd886ffe91e...,60bf2fa6eef998012a1deba2f9b7153ff5de9138322923...
1,consensus_reversal,COMPLETED,8388,304006,7,0.571429,-0.3641,True,bundle:5036382a936772fc01e50e9a36a2fd886ffe91e...,93caad41e57a933b78e910c80aa0b08895c9f7e2360f9c...


## Decision funnel and refusal metadata

In [3]:
gate_rows = []
for run in analysis['runs']:
    total = run['decisions']['evaluations']
    for gate, count in run['decisions'].get('gate_counts', {}).items():
        gate_rows.append({
            'run': run['label'], 'gate': gate, 'evaluations': count,
            'share': count / total if total else None,
            'candidate_yes': run['decisions'].get('candidate_outcomes', {}).get('yes', 0),
            'candidate_no': run['decisions'].get('candidate_outcomes', {}).get('no', 0),
        })
pd.DataFrame(gate_rows).sort_values(['run', 'evaluations'], ascending=[True, False])

,run,gate,evaluations,share,candidate_yes,candidate_no
0,baseline,entry_window,6637,0.793425,481,943
1,baseline,model_market_divergence,699,0.083562,481,943
2,baseline,min_probability,662,0.079139,481,943
3,baseline,minimum_time_to_close,192,0.022953,481,943
4,baseline,candle_freshness,112,0.013389,481,943
5,baseline,settlement_margin,31,0.003706,481,943
6,baseline,min_edge,30,0.003586,481,943
7,baseline,live_edge,2,0.000239,481,943
8,consensus_reversal,entry_window,6660,0.793991,614,835
9,consensus_reversal,min_probability,894,0.106581,614,835


## Strategy results

In [4]:
strategy = pd.DataFrame(analysis['aggregate']['strategy_lab'])
strategy.sort_values(['net_pnl', 'closed'], ascending=[False, False])

,strategy,runs_present,closed,wins,losses,win_rate,net_pnl,pnl_per_closed_position
4,consensus_tail_carry,2,10,10,0,1.000000,4.2635,0.426350
2,complementary_pair_arb,2,0,0,0,NaN,0.0000,NaN
14,trend_trailing,2,0,0,0,NaN,0.0000,NaN
5,displaced_shelf,2,6,2,4,0.333333,-26.1044,-4.350733
3,confirmed_reversal_exit,1,20,8,12,0.400000,-29.0491,-1.452455
9,principal_recovery,1,18,9,9,0.500000,-32.2088,-1.789378
1,change_point_flip,2,16,8,8,0.500000,-32.3553,-2.022206
15,winning_hold_take_profit,1,18,9,9,0.500000,-33.0186,-1.834367
13,strike_distance,2,14,5,9,0.357143,-42.0211,-3.001507
10,probability_acceleration,2,28,10,18,0.357143,-54.7115,-1.953982


## Matched exit-policy comparison

In [5]:
exit_rows = []
for run in analysis['runs']:
    for row in run['matched_exit_comparison']:
        exit_rows.append({'run': run['label'], **row})
pd.DataFrame(exit_rows)

,run,strategy,matched_positions,actual_pnl,hold_counterfactual_pnl,delta_vs_hold,winner_markets,winner_upside_capture,winner_pnl,winner_hold_pnl,loser_markets,loser_exit_pnl,loser_hold_pnl,loss_avoided
0,baseline,fixed_take_profit,26,-56.1030,-12.6276,-43.4754,13,0.271187,21.5068,79.3062,13,-77.6098,-91.9338,14.3240
1,baseline,adaptive_take_profit,22,-51.0482,-46.7333,-4.3149,9,0.904539,40.8856,45.2005,13,-91.9338,-91.9338,0.0000
2,consensus_reversal,fixed_take_profit,22,-42.1656,-11.4005,-30.7651,13,0.396307,27.3867,69.1047,9,-69.5523,-80.5052,10.9529
3,consensus_reversal,adaptive_take_profit,18,-37.5440,-31.1039,-6.4401,9,0.869637,42.9612,49.4013,9,-80.5052,-80.5052,0.0000


## Main-paper trade decisions

In [6]:
detail = []
for run in analysis['runs']:
    detail.extend({'run': run['label'], **row} for row in run['main_trade_details'])
pd.DataFrame(detail)

,run,trade_id,asset,ticker,outcome,status,exchange_status,placed_at,exited_at,contracts,entry_price,exit_price,entry_probability,pnl,maximum_pnl_if_winner,positive_exit_capture_ratio
0,baseline,d8c1dfe1-f80d-467c-adda-f2fa0a1cf302,BTC,KXBTC15M-26AUG241815-15,no,EXITED,take_profit,2026-08-24T22:01:28.527185+00:00,2026-08-24T22:11:18.502282+00:00,13.00,0.74,0.943,0.806577,2.4096,3.20,0.753000
1,baseline,63c06982-b5a7-4a13-84d7-8feb2094d33d,BTC,KXBTC15M-26AUG241845-45,yes,EXITED,take_profit,2026-08-24T22:37:19.273159+00:00,2026-08-24T22:44:28.302073+00:00,12.00,0.81,0.981,0.861216,1.9052,2.15,0.886140
2,baseline,067d2d9f-1417-4f50-955a-cc4d21c6b5f4,GOLD,KXGOLD15M-26AUG241900-00,yes,EXITED,take_profit,2026-08-24T22:51:19.934892+00:00,2026-08-24T22:52:08.442441+00:00,13.00,0.70,0.936,0.807859,2.8134,3.70,0.760378
3,baseline,420ddaa0-37d4-4c06-bb7a-ca5a5564ff2b,BTC,KXBTC15M-26AUG241900-00,no,EXITED,take_profit,2026-08-24T22:56:08.583329+00:00,2026-08-24T22:58:28.333525+00:00,12.00,0.77,0.951,0.820214,1.9824,2.61,0.759540
4,baseline,6617ddb4-1e9a-4f36-88e6-0b4b7906d96f,BTC,KXBTC15M-26AUG241915-15,no,WON,determined,2026-08-24T23:03:08.514446+00:00,NaN,13.00,0.73,NaN,0.822991,3.3300,3.33,1.000000
5,baseline,5be357a1-3690-4b9a-9227-844d4427ec8c,BTC,KXBTC15M-26AUG241945-45,yes,EXITED,take_profit,2026-08-24T23:41:28.465310+00:00,2026-08-24T23:43:58.397295+00:00,12.00,0.79,0.992,0.825977,2.2756,2.38,0.956134
6,consensus_reversal,2c4f828c-810b-48a3-9e7e-1f36dd797524,BTC,KXBTC15M-26AUG242100-00,yes,EXITED,take_profit,2026-08-25T00:46:08.465462+00:00,2026-08-25T00:52:18.485531+00:00,12.00,0.77,0.955,0.830032,2.0328,2.61,0.778851
7,consensus_reversal,56491aa0-6a5a-4cca-80b7-c0ac9ee1d48b,BTC,KXBTC15M-26AUG242115-15,yes,EXITED,confirmed_reversal,2026-08-25T01:03:28.505929+00:00,2026-08-25T01:10:38.459743+00:00,13.00,0.71,0.220,0.759106,-6.7173,3.58,NaN
8,consensus_reversal,68750ff9-a96f-44b8-9552-11b1cd764f94,BTC,KXBTC15M-26AUG242130-30,yes,WON,finalized,2026-08-25T01:21:08.431001+00:00,NaN,13.00,0.72,NaN,0.759936,3.4500,3.45,1.000000
9,consensus_reversal,f14fc85a-be18-4985-9d77-ea3222ec4bd6,BTC,KXBTC15M-26AUG242145-45,no,EXITED,confirmed_reversal,2026-08-25T01:33:18.466822+00:00,2026-08-25T01:38:48.455352+00:00,13.00,0.72,0.550,0.769824,-2.6262,3.45,NaN


## Model diagnostics

In [7]:
models = pd.DataFrame(analysis['models'])
models.sort_values('auc', ascending=False)

,asset,auc,baseline_auc,indicator_delta_auc,deployment_status,weight
3,HYPE,0.559809,0.513889,0.045920,active,0.089714
6,SOL,0.532485,0.520224,0.012262,shadow_only,0.000000
5,SILVER,0.526142,0.521152,0.004991,shadow_only,0.000000
1,DOGE,0.523349,0.491560,0.031789,shadow_only,0.000000
7,WTI,0.520297,0.493425,0.026872,shadow_only,0.000000
2,GOLD,0.512484,0.550796,-0.038312,shadow_only,0.000000
9,ZEC,0.508488,0.521338,-0.012850,shadow_only,0.000000
4,NEAR,0.507041,0.492761,0.014280,shadow_only,0.000000
8,XRP,0.497939,0.487770,0.010169,shadow_only,0.000000
0,BNB,0.464392,0.435444,0.028949,shadow_only,0.000000


## Limitations and takeaways

- The runs cover different market windows and configuration hashes; cross-run differences are descriptive, not causal. The same-window strategy-lab comparison is the strongest available evidence.
- A high win rate can coexist with negative P&L when losses dominate many small wins. Promotion therefore requires positive fee-adjusted expectancy, adequate sample size, and stable results across independent windows.
- Model AUCs near 0.50 imply weak ranking power. Only artifacts that cleared the deployment threshold were allowed a non-zero blend weight.
- No result guarantees live profitability; fill latency, queue position, and partial fills remain material execution risks.